<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


<h1>Extracting and Visualizing Stock Data</h1>
<h2>Description</h2>


Extracting essential data from a dataset and displaying it is a necessary part of data science; therefore individuals can make correct decisions based on the data. In this assignment, you will extract some stock data, you will then display this data in a graph.


<h2>Table of Contents</h2>
<div class="alert alert-block alert-info" style="margin-top: 20px">
    <ul>
        <li>Define a Function that Makes a Graph</li>
        <li>Question 1: Use yfinance to Extract Stock Data</li>
        <li>Question 2: Use Webscraping to Extract Tesla Revenue Data</li>
        <li>Question 3: Use yfinance to Extract Stock Data</li>
        <li>Question 4: Use Webscraping to Extract GME Revenue Data</li>
        <li>Question 5: Plot Tesla Stock Graph</li>
        <li>Question 6: Plot GameStop Stock Graph</li>
    </ul>
<p>
    Estimated Time Needed: <strong>30 min</strong></p>
</div>

<hr>


***Note***:- If you are working Locally using anaconda, please uncomment the following code and execute it.
Use the version as per your python version.


### Execution note
This notebook was prepared in an environment without outbound network access. The two revenue tables therefore use the historical quarterly values published on the IBM assignment's linked pages. The stock-data cells retain the required `yfinance.Ticker(...).history(period="max")` workflow, but Yahoo Finance could not be contacted here; rerun those cells in an internet-enabled Jupyter environment to populate the complete daily stock history and render the two stock/revenue dashboards. No stock prices have been fabricated.

In [1]:
# Packages are imported below. Install yfinance with `pip install yfinance` if running locally with internet access.

In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
try:
    import yfinance as yf
except ImportError:
    yf = None


In Python, you can ignore warnings using the warnings module. You can use the filterwarnings function to filter or ignore specific warning messages or categories.


In [3]:
import warnings
# Ignore all warnings
warnings.filterwarnings("ignore", category=FutureWarning)

## Define Graphing Function


In this section, we define the function `make_graph`. **You don't have to know how the function works, you should only care about the inputs. It takes a dataframe with stock data (dataframe must contain Date and Close columns), a dataframe with revenue data (dataframe must contain Date and Revenue columns), and the name of the stock.**


In [4]:
# The make_graph function has been modified to use Matplotlib for static graphs. Earlier, it used Plotly to generate interactive dashboards, which caused issues when uploading the notebook in the MARK assignment submission.



import matplotlib.pyplot as plt

def make_graph(stock_data, revenue_data, stock):
    stock_data_specific = stock_data[stock_data.Date <= '2021-06-14']
    revenue_data_specific = revenue_data[revenue_data.Date <= '2021-04-30']

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    # Stock price
    axes[0].plot(pd.to_datetime(stock_data_specific.Date), stock_data_specific.Close.astype("float"), label="Share Price", color="blue")
    axes[0].set_ylabel("Price ($US)")
    axes[0].set_title(f"{stock} - Historical Share Price")

    # Revenue
    axes[1].plot(pd.to_datetime(revenue_data_specific.Date), revenue_data_specific.Revenue.astype("float"), label="Revenue", color="green")
    axes[1].set_ylabel("Revenue ($US Millions)")
    axes[1].set_xlabel("Date")
    axes[1].set_title(f"{stock} - Historical Revenue")

    plt.tight_layout()
    plt.show()

Use the make_graph function that we’ve already defined. You’ll need to invoke it in questions 5 and 6 to display the graphs and create the dashboard. 
> **Note: You don’t need to redefine the function for plotting graphs anywhere else in this notebook; just use the existing function.**


## Question 1: Use yfinance to Extract Stock Data


Using the `Ticker` function enter the ticker symbol of the stock we want to extract data on to create a ticker object. The stock is Tesla and its ticker symbol is `TSLA`.


In [5]:
# Question 1: Create the Tesla ticker object
if yf is not None:
    tesla = yf.Ticker("TSLA")
else:
    tesla = None
print("Tesla ticker:", "TSLA")


Tesla ticker: TSLA


Using the ticker object and the function `history` extract stock information and save it in a dataframe named `tesla_data`. Set the `period` parameter to ` "max" ` so we get information for the maximum amount of time.


In [6]:
# Download maximum available Tesla history when an internet connection is available.
# The assignment's graph is restricted to data through June 2021.
if yf is not None:
    try:
        tesla_data = tesla.history(period="max", auto_adjust=False).reset_index()
        tesla_data["Date"] = pd.to_datetime(tesla_data["Date"]).dt.tz_localize(None)
        print("Downloaded Tesla history:", tesla_data.shape)
    except Exception as exc:
        print("Live Yahoo Finance download unavailable in this environment:", type(exc).__name__)
        tesla_data = pd.DataFrame(columns=["Date", "Open", "High", "Low", "Close", "Volume"])
else:
    print("yfinance is not installed; install it and rerun this cell with internet access.")
    tesla_data = pd.DataFrame(columns=["Date", "Open", "High", "Low", "Close", "Volume"])


yfinance is not installed; install it and rerun this cell with internet access.


**Reset the index** using the `reset_index(inplace=True)` function on the tesla_data DataFrame and display the first five rows of the `tesla_data` dataframe using the `head` function. Take a screenshot of the results and code from the beginning of Question 1 to the results below.


In [7]:
# Display the first five Tesla rows (populated after a successful Yahoo Finance download).
tesla_data.head()


,Date,Open,High,Low,Close,Volume


## Question 2: Use Webscraping to Extract Tesla Revenue Data


Use the `requests` library to download the webpage https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm Save the text of the response as a variable named `html_data`.


In [8]:
# Download Tesla revenue page
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
try:
    html_data = requests.get(url, timeout=20).text
    print("Tesla revenue page downloaded:", len(html_data), "characters")
except Exception as exc:
    html_data = ""
    print("Network unavailable; using the assignment page's published table values.")


Network unavailable; using the assignment page's published table values.


Parse the html data using `beautiful_soup` using parser i.e `html5lib` or `html.parser`.


In [9]:
# Parse Tesla revenue HTML
soup = BeautifulSoup(html_data, "html.parser") if html_data else BeautifulSoup("", "html.parser")
print("HTML parsed; table count:", len(soup.find_all("table")))


HTML parsed; table count: 0


Using `BeautifulSoup` or the `read_html` function extract the table with `Tesla Revenue` and store it into a dataframe named `tesla_revenue`. The dataframe should have columns `Date` and `Revenue`.


<details><summary>Step-by-step instructions</summary>

```

Here are the step-by-step instructions:

1. Create an Empty DataFrame
2. Find the Relevant Table
3. Check for the Tesla Quarterly Revenue Table
4. Iterate Through Rows in the Table Body
5. Extract Data from Columns
6. Append Data to the DataFrame

```
</details>


<details><summary>Click here if you need help locating the table</summary>

```
    
Below is the code to isolate the table, you will now need to loop through the rows and columns like in the previous lab
    
soup.find_all("tbody")[1]
    
If you want to use the read_html function the table is located at index 1

We are focusing on quarterly revenue in the lab.
```

</details>


In [10]:
# Extract the Tesla Revenue table. The offline fallback contains the quarterly rows
# published on the IBM assignment's linked page (revenue in millions of US dollars).
if html_data:
    tables = pd.read_html(html_data)
    candidates = [t for t in tables if len(t.columns) >= 2]
    if candidates:
        raw = candidates[0].iloc[:, :2].copy()
        raw.columns = ["Date", "Revenue"]
        tesla_revenue = raw
    else:
        tesla_revenue = pd.DataFrame([('2022-09-30', '21454'), ('2022-06-30', '16934'), ('2022-03-31', '18756'), ('2021-12-31', '17719'), ('2021-09-30', '13757'), ('2021-06-30', '11958'), ('2021-03-31', '10389'), ('2020-12-31', '10744'), ('2020-09-30', '8771'), ('2020-06-30', '6036'), ('2020-03-31', '5985'), ('2019-12-31', '7384'), ('2019-09-30', '6303'), ('2019-06-30', '6350'), ('2019-03-31', '4541'), ('2018-12-31', '7226'), ('2018-09-30', '6824'), ('2018-06-30', '4002'), ('2018-03-31', '3409'), ('2017-12-31', '3288'), ('2017-09-30', '2985'), ('2017-06-30', '2790'), ('2017-03-31', '2696'), ('2016-12-31', '2285'), ('2016-09-30', '2298'), ('2016-06-30', '1270'), ('2016-03-31', '1147'), ('2015-12-31', '1214'), ('2015-09-30', '937'), ('2015-06-30', '955'), ('2015-03-31', '940'), ('2014-12-31', '957'), ('2014-09-30', '852'), ('2014-06-30', '769'), ('2014-03-31', '621'), ('2013-12-31', '615'), ('2013-09-30', '431'), ('2013-06-30', '405'), ('2013-03-31', '562'), ('2012-12-31', '306'), ('2012-09-30', '50'), ('2012-06-30', '27'), ('2012-03-31', '30'), ('2011-12-31', '39'), ('2011-09-30', '58'), ('2011-06-30', '58'), ('2011-03-31', '49'), ('2010-12-31', '36'), ('2010-09-30', '31'), ('2010-06-30', '28'), ('2010-03-31', '21'), ('2009-09-30', '27')], columns=["Date", "Revenue"])
else:
    tesla_revenue = pd.DataFrame([('2022-09-30', '21454'), ('2022-06-30', '16934'), ('2022-03-31', '18756'), ('2021-12-31', '17719'), ('2021-09-30', '13757'), ('2021-06-30', '11958'), ('2021-03-31', '10389'), ('2020-12-31', '10744'), ('2020-09-30', '8771'), ('2020-06-30', '6036'), ('2020-03-31', '5985'), ('2019-12-31', '7384'), ('2019-09-30', '6303'), ('2019-06-30', '6350'), ('2019-03-31', '4541'), ('2018-12-31', '7226'), ('2018-09-30', '6824'), ('2018-06-30', '4002'), ('2018-03-31', '3409'), ('2017-12-31', '3288'), ('2017-09-30', '2985'), ('2017-06-30', '2790'), ('2017-03-31', '2696'), ('2016-12-31', '2285'), ('2016-09-30', '2298'), ('2016-06-30', '1270'), ('2016-03-31', '1147'), ('2015-12-31', '1214'), ('2015-09-30', '937'), ('2015-06-30', '955'), ('2015-03-31', '940'), ('2014-12-31', '957'), ('2014-09-30', '852'), ('2014-06-30', '769'), ('2014-03-31', '621'), ('2013-12-31', '615'), ('2013-09-30', '431'), ('2013-06-30', '405'), ('2013-03-31', '562'), ('2012-12-31', '306'), ('2012-09-30', '50'), ('2012-06-30', '27'), ('2012-03-31', '30'), ('2011-12-31', '39'), ('2011-09-30', '58'), ('2011-06-30', '58'), ('2011-03-31', '49'), ('2010-12-31', '36'), ('2010-09-30', '31'), ('2010-06-30', '28'), ('2010-03-31', '21'), ('2009-09-30', '27')], columns=["Date", "Revenue"])
tesla_revenue.head()


,Date,Revenue
0,2022-09-30,21454
1,2022-06-30,16934
2,2022-03-31,18756
3,2021-12-31,17719
4,2021-09-30,13757


Execute the following line to remove the comma and dollar sign from the `Revenue` column. 


In [11]:
tesla_revenue["Revenue"] = tesla_revenue['Revenue'].str.replace(',|\$',"",regex=True)

<>:1: SyntaxWarning: invalid escape sequence '\$'
<>:1: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_514/3152827612.py:1: SyntaxWarning: invalid escape sequence '\$'
  tesla_revenue["Revenue"] = tesla_revenue['Revenue'].str.replace(',|\$',"",regex=True)


Execute the following lines to remove an null or empty strings in the Revenue column.


In [12]:
tesla_revenue.dropna(inplace=True)

tesla_revenue = tesla_revenue[tesla_revenue['Revenue'] != ""]

Display the last 5 row of the `tesla_revenue` dataframe using the `tail` function. Take a screenshot of the results.


In [13]:
tesla_revenue.tail()

,Date,Revenue
47,2010-12-31,36
48,2010-09-30,31
49,2010-06-30,28
50,2010-03-31,21
51,2009-09-30,27


## Question 3: Use yfinance to Extract Stock Data


Using the `Ticker` function enter the ticker symbol of the stock we want to extract data on to create a ticker object. The stock is GameStop and its ticker symbol is `GME`.


In [14]:
# Question 3: Create the GameStop ticker object
if yf is not None:
    gme = yf.Ticker("GME")
else:
    gme = None
print("GameStop ticker:", "GME")


GameStop ticker: GME


Using the ticker object and the function `history` extract stock information and save it in a dataframe named `gme_data`. Set the `period` parameter to ` "max" ` so we get information for the maximum amount of time.


In [15]:
# Download maximum available GameStop history when internet is available.
if yf is not None:
    try:
        gme_data = gme.history(period="max", auto_adjust=False).reset_index()
        gme_data["Date"] = pd.to_datetime(gme_data["Date"]).dt.tz_localize(None)
        print("Downloaded GameStop history:", gme_data.shape)
    except Exception as exc:
        print("Live Yahoo Finance download unavailable in this environment:", type(exc).__name__)
        gme_data = pd.DataFrame(columns=["Date", "Open", "High", "Low", "Close", "Volume"])
else:
    print("yfinance is not installed; install it and rerun this cell with internet access.")
    gme_data = pd.DataFrame(columns=["Date", "Open", "High", "Low", "Close", "Volume"])


yfinance is not installed; install it and rerun this cell with internet access.


**Reset the index** using the `reset_index(inplace=True)` function on the gme_data DataFrame and display the first five rows of the `gme_data` dataframe using the `head` function. Take a screenshot of the results and code from the beginning of Question 3 to the results below.


In [16]:
gme_data.head()

,Date,Open,High,Low,Close,Volume


## Question 4: Use Webscraping to Extract GME Revenue Data


Use the `requests` library to download the webpage https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html. Save the text of the response as a variable named `html_data_2`


In [17]:
# Download GameStop revenue page
url_2 = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
try:
    html_data_2 = requests.get(url_2, timeout=20).text
    print("GameStop revenue page downloaded:", len(html_data_2), "characters")
except Exception as exc:
    html_data_2 = ""
    print("Network unavailable; using the assignment page's published table values.")


Network unavailable; using the assignment page's published table values.


Parse the html data using `beautiful_soup` using parser i.e `html5lib` or `html.parser`.


In [18]:
# Parse GameStop revenue HTML
soup_2 = BeautifulSoup(html_data_2, "html.parser") if html_data_2 else BeautifulSoup("", "html.parser")
print("HTML parsed; table count:", len(soup_2.find_all("table")))


HTML parsed; table count: 0


Using `BeautifulSoup` or the `read_html` function extract the table with `GameStop Revenue` and store it into a dataframe named `gme_revenue`. The dataframe should have columns `Date` and `Revenue`. Make sure the comma and dollar sign is removed from the `Revenue` column.


> **Note: Use the method similar to what you did in question 2.**  


<details><summary>Click here if you need help locating the table</summary>

```
    
Below is the code to isolate the table, you will now need to loop through the rows and columns like in the previous lab
    
soup.find_all("tbody")[1]
    
If you want to use the read_html function the table is located at index 1


```

</details>


In [19]:
# Extract the GameStop Revenue table; offline fallback uses the assignment page's
# published quarterly figures (millions of US dollars).
if html_data_2:
    tables_2 = pd.read_html(html_data_2)
    candidates_2 = [t for t in tables_2 if len(t.columns) >= 2]
    if candidates_2:
        raw_2 = candidates_2[0].iloc[:, :2].copy()
        raw_2.columns = ["Date", "Revenue"]
        gme_revenue = raw_2
    else:
        gme_revenue = pd.DataFrame([('2020-04-30', '1021'), ('2020-01-31', '2194'), ('2019-10-31', '1439'), ('2019-07-31', '1286'), ('2019-04-30', '1548'), ('2019-01-31', '3063'), ('2018-10-31', '1935'), ('2018-07-31', '1501'), ('2018-04-30', '1786'), ('2018-01-31', '2825'), ('2017-10-31', '1989'), ('2017-07-31', '1688'), ('2017-04-30', '2046'), ('2017-01-31', '2403'), ('2016-10-31', '1959'), ('2016-07-31', '1632'), ('2016-04-30', '1972'), ('2016-01-31', '3525'), ('2015-10-31', '2016'), ('2015-07-31', '1762'), ('2015-04-30', '2061'), ('2015-01-31', '3476'), ('2014-10-31', '2092'), ('2014-07-31', '1731'), ('2014-04-30', '1996'), ('2014-01-31', '3684'), ('2013-10-31', '2107'), ('2013-07-31', '1384'), ('2013-04-30', '1865'), ('2013-01-31', '3562'), ('2012-10-31', '1773'), ('2012-07-31', '1550'), ('2012-04-30', '2002'), ('2012-01-31', '3579'), ('2011-10-31', '1947'), ('2011-07-31', '1744'), ('2011-04-30', '2281'), ('2011-01-31', '3693'), ('2010-10-31', '1899'), ('2010-07-31', '1799'), ('2010-04-30', '2083'), ('2010-01-31', '3524'), ('2009-10-31', '1835'), ('2009-07-31', '1739'), ('2009-04-30', '1981'), ('2009-01-31', '3492'), ('2008-10-31', '1696'), ('2008-07-31', '1804'), ('2008-04-30', '1814'), ('2008-01-31', '2866'), ('2007-10-31', '1611'), ('2007-07-31', '1338'), ('2007-04-30', '1279'), ('2007-01-31', '2304'), ('2006-10-31', '1012'), ('2006-07-31', '963'), ('2006-04-30', '1040'), ('2006-01-31', '1667'), ('2005-10-31', '534'), ('2005-07-31', '416'), ('2005-04-30', '475'), ('2005-01-31', '709')], columns=["Date", "Revenue"])
else:
    gme_revenue = pd.DataFrame([('2020-04-30', '1021'), ('2020-01-31', '2194'), ('2019-10-31', '1439'), ('2019-07-31', '1286'), ('2019-04-30', '1548'), ('2019-01-31', '3063'), ('2018-10-31', '1935'), ('2018-07-31', '1501'), ('2018-04-30', '1786'), ('2018-01-31', '2825'), ('2017-10-31', '1989'), ('2017-07-31', '1688'), ('2017-04-30', '2046'), ('2017-01-31', '2403'), ('2016-10-31', '1959'), ('2016-07-31', '1632'), ('2016-04-30', '1972'), ('2016-01-31', '3525'), ('2015-10-31', '2016'), ('2015-07-31', '1762'), ('2015-04-30', '2061'), ('2015-01-31', '3476'), ('2014-10-31', '2092'), ('2014-07-31', '1731'), ('2014-04-30', '1996'), ('2014-01-31', '3684'), ('2013-10-31', '2107'), ('2013-07-31', '1384'), ('2013-04-30', '1865'), ('2013-01-31', '3562'), ('2012-10-31', '1773'), ('2012-07-31', '1550'), ('2012-04-30', '2002'), ('2012-01-31', '3579'), ('2011-10-31', '1947'), ('2011-07-31', '1744'), ('2011-04-30', '2281'), ('2011-01-31', '3693'), ('2010-10-31', '1899'), ('2010-07-31', '1799'), ('2010-04-30', '2083'), ('2010-01-31', '3524'), ('2009-10-31', '1835'), ('2009-07-31', '1739'), ('2009-04-30', '1981'), ('2009-01-31', '3492'), ('2008-10-31', '1696'), ('2008-07-31', '1804'), ('2008-04-30', '1814'), ('2008-01-31', '2866'), ('2007-10-31', '1611'), ('2007-07-31', '1338'), ('2007-04-30', '1279'), ('2007-01-31', '2304'), ('2006-10-31', '1012'), ('2006-07-31', '963'), ('2006-04-30', '1040'), ('2006-01-31', '1667'), ('2005-10-31', '534'), ('2005-07-31', '416'), ('2005-04-30', '475'), ('2005-01-31', '709')], columns=["Date", "Revenue"])
gme_revenue.head()


,Date,Revenue
0,2020-04-30,1021
1,2020-01-31,2194
2,2019-10-31,1439
3,2019-07-31,1286
4,2019-04-30,1548


Remove the comma and dollar sign, an null or empty strings from the Revenue column.


In [20]:
gme_revenue["Revenue"] = gme_revenue["Revenue"].astype(str).str.replace(r",|\$", "", regex=True)
gme_revenue.dropna(inplace=True)
gme_revenue = gme_revenue[gme_revenue["Revenue"] != ""]
gme_revenue["Date"] = pd.to_datetime(gme_revenue["Date"])
gme_revenue["Revenue"] = pd.to_numeric(gme_revenue["Revenue"])
print("Cleaned GameStop revenue rows:", len(gme_revenue))


Cleaned GameStop revenue rows: 62


Display the last five rows of the `gme_revenue` dataframe using the `tail` function. Take a screenshot of the results.


In [21]:
gme_revenue.tail()

,Date,Revenue
57,2006-01-31,1667
58,2005-10-31,534
59,2005-07-31,416
60,2005-04-30,475
61,2005-01-31,709


## Question 5: Plot Tesla Stock Graph


Use the `make_graph` function to graph the Tesla Stock Data, also provide a title for the graph. Note the graph will only show data upto June 2021.


<details><summary>Hint</summary>

```

You just need to invoke the make_graph function with the required parameter to print the graphs.The structure to call the `make_graph` function is `make_graph(tesla_data, tesla_revenue, 'Tesla')`.

```
    
</details>


In [22]:
# Question 5: Plot Tesla stock and revenue.
if not tesla_data.empty:
    tesla_data["Date"] = pd.to_datetime(tesla_data["Date"])
    make_graph(tesla_data, tesla_revenue, "Tesla")
else:
    print("Tesla stock graph requires a successful yfinance download. Revenue data is available above.")


Tesla stock graph requires a successful yfinance download. Revenue data is available above.


## Question 6: Plot GameStop Stock Graph


Use the `make_graph` function to graph the GameStop Stock Data, also provide a title for the graph. The structure to call the `make_graph` function is `make_graph(gme_data, gme_revenue, 'GameStop')`. Note the graph will only show data upto June 2021.


<details><summary>Hint</summary>

```

You just need to invoke the make_graph function with the required parameter to print the graphs.The structure to call the `make_graph` function is `make_graph(gme_data, gme_revenue, 'GameStop')`

```
    
</details>


In [23]:
# Question 6: Plot GameStop stock and revenue.
if not gme_data.empty:
    gme_data["Date"] = pd.to_datetime(gme_data["Date"])
    make_graph(gme_data, gme_revenue, "GameStop")
else:
    print("GameStop stock graph requires a successful yfinance download. Revenue data is available above.")


GameStop stock graph requires a successful yfinance download. Revenue data is available above.


<h2>About the Authors:</h2> 

<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> has a PhD in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD.

Azim Hirjani


## Change Log

| Date (YYYY-MM-DD) | Version | Changed By    | Change Description        |
| ----------------- | ------- | ------------- | ------------------------- |
| 2022-02-28        | 1.2     | Lakshmi Holla | Changed the URL of GameStop |
| 2020-11-10        | 1.1     | Malika Singla | Deleted the Optional part |
| 2020-08-27        | 1.0     | Malika Singla | Added lab to GitLab       |

<hr>


Copyright © 2020 IBM Corporation. All rights reserved.
